# Run Any Kind of OLS Regression (ANOVA, GLM, etc.)

### Authors: Calvin Howard.

#### Last updated: July 6, 2023

Use this to run/test a statistical model (e.g., regression or T-tests) on a spreadsheet.

Notes:
- To best use this notebook, you should be familar with GLM design and Contrast Matrix design. See this webpage to get started:
[FSL's GLM page](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/GLM)

# 00 - Import CSV with All Data
**The CSV is expected to be in this format**
- ID and absolute paths to niftis are critical
```
+-----+----------------------------+--------------+--------------+--------------+
| ID  | Nifti_File_Path            | Covariate_1  | Covariate_2  | Covariate_3  |
+-----+----------------------------+--------------+--------------+--------------+
| 1   | /path/to/file1.nii.gz      | 0.5          | 1.2          | 3.4          |
| 2   | /path/to/file2.nii.gz      | 0.7          | 1.4          | 3.1          |
| 3   | /path/to/file3.nii.gz      | 0.6          | 1.5          | 3.5          |
| 4   | /path/to/file4.nii.gz      | 0.9          | 1.1          | 3.2          |
| ... | ...                        | ...          | ...          | ...          |
+-----+----------------------------+--------------+--------------+--------------+
```

Prep Output Direction

In [ ]:
# Specify where you want to save your results to
out_dir = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/acoe_reliability/results/ICC'

Import Data

In [ ]:
# Specify the path to your CSV file containing NIFTI paths
input_csv_path = '/Users/cu135/Partners HealthCare Dropbox/Calvin Howard/studies/acoe_reliability/scores_v1_combined.csv'
sheet = None

In [ ]:
from calvin_utils.permutation_analysis_utils.statsmodels_palm import CalvinStatsmodelsPalm
# Instantiate the PalmPrepararation class
cal_palm = CalvinStatsmodelsPalm(input_csv_path=input_csv_path, output_dir=out_dir, sheet=sheet)
# Call the process_nifti_paths method
data_df = cal_palm.read_and_display_data()
display(data_df)

# 01 - Preprocess Your Data

**Handle NANs**
- Set drop_nans=True is you would like to remove NaNs from data
- Provide a column name or a list of column names to remove NaNs from

In [ ]:
data_df.columns

In [ ]:
drop_list = ['Q2']

In [ ]:
data_df = cal_palm.drop_nans_from_columns(columns_to_drop_from=drop_list)
display(data_df)

**Drop Row Based on Value of Column**

Define the column, condition, and value for dropping rows
- column = 'your_column_name'
- condition = 'above'  # Options: 'equal', 'above', 'below'

In [ ]:
data_df.columns

Set the parameters for dropping rows

In [ ]:
column = 'origin'  # The column you'd like to evaluate
condition = 'not'  # The condition to check ('equal', 'above', 'below', 'not')
value = 'president@cog-net.com' # The value to drop if found

In [ ]:
data_df, other_df = cal_palm.drop_rows_based_on_value(column, condition, value)
display(data_df)

**Standardize Data**
- Enter Columns you Don't want to standardize into a list

In [ ]:
# Remove anything you don't want to standardize
# cols_not_to_standardize = None # ['Z_Scored_Percent_Cognitive_Improvement_By_Origin_Group', 'Z_Scored_Subiculum_T_By_Origin_Group_'] #['Age']

In [ ]:
# data_df = cal_palm.standardize_columns(cols_not_to_standardize)
# data_df

In [ ]:
# for col in data_df.columns:
#     if 'CSF' and 'eh' not in col:
#         data_df[col] = data_df[col] * -1

Melt Df

In [ ]:
import pandas as pd

def stack_A_M_wide(df: pd.DataFrame) -> pd.DataFrame:
    id_cols = [
        col for col in df.columns
        if not col.endswith("A") and not col.endswith("M")
    ]

    a_cols = [col for col in df.columns if col.endswith("A")]
    m_cols = [col for col in df.columns if col.endswith("M")]

    df_a = df[id_cols + a_cols].copy()
    df_m = df[id_cols + m_cols].copy()

    df_a = df_a.rename(columns={col: col[:-1] for col in a_cols})
    df_m = df_m.rename(columns={col: col[:-1] for col in m_cols})

    df_a["type"] = "A"
    df_m["type"] = "M"

    long_wide_df = pd.concat([df_a, df_m], axis=0, ignore_index=True)

    question_cols = sorted(
        [col for col in long_wide_df.columns if col.startswith("Q")],
        key=lambda x: int(x[1:])
    )

    final_cols = id_cols + ["type"] + question_cols

    if "Total" in long_wide_df.columns:
        final_cols += ["Total"]

    return long_wide_df[final_cols]
data_df = stack_A_M_wide(data_df)
data_df

# 02 - Derive ICCs

In [ ]:
columns_to_compare = ['Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9',
       'Q10', 'Q11', 'Q12', 'Q13', 'Q14', 'Q15', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Total']
rater_column = 'type'
y_label = 'Question'

In [ ]:
from calvin_utils.statistical_utils.icc_analysis import ICCAnalysis
# Initialize the class
analysis = ICCAnalysis()
# Orchestrate multiple iterations of t-tests and plot boxplots
analysis.orchestrate_icc_analysis(data_df, rater_column, columns_to_compare, outdir=out_dir, icc="ICC(1,1)")